# Overture MK2 Closure Sizing
Edited 10/6/2026, Joris Lindinger

Based on OVMK1 master document

## General data

Thin-walled Assumption *not* valid, since diameter/thickness !> 20

In [ ]:
import math
import numpy as np

# geometry data
d_inner = 4 #inches
t_wall = 0.25 #inches
l = 12 #inches

# material data (304 stainless) STANDARD TEMPERATURE FOR NOW, CHECK AT HIGH T AGAIN!!!
sigma_yield = 29732.7 #yield strength, psi
tau_yield = 0.577*sigma_yield #shear yield strength based on van mises, psi

# other data
p_chamber = 300 #psi
n_safety = 2 #safety factor 2, from OVMK1 documentation

## Hoop stress

Thick-walled formula

r =^ radius at point of interest<br>
A, B =^ boundary conditions

P_a =^ internal pressure<br>
P_b =^ external pressure<br>
a =^ inner radius<br>
b =^ outer radius<br>

In [ ]:
P_a = p_chamber
P_b = 14.7 #psi, sea level assumed for now

a = d_inner/2
b = a + t_wall

r = a #using inner radius for worst-case estimate

A = (P_a*a**2 - P_b*b**2)/(b**2-a**2)
B = a**2*b**2*(P_a-P_b)/(b**2 - a**2)


sigma_hoop = A + B / r**2

print(f"Hoop stress: {sigma_hoop} psi")

Hoop stress: 2418.741176470588 psi
Yield strength: 29732.7 psi
Safety factor: 12.292633990456874


# Axial stress

This was somehow not included in the OVMK1 master document

Is uniformly distributed across the wall, so formula is simple

In [ ]:
sigma_axial = (P_a*a**2-P_b*b**2)/(b**2-a**2)
print(f"Axial stress: {sigma_axial} psi")

## Radial stress

A, B, r same as for hoop stress

In [ ]:
sigma_rad = A - B / r**2

print(f"Radial stress: {sigma_rad} psi")

Radial stress: -300.0 psi
Yield strength: 29732.7 psi
Safety factor: 99.10900000000001


## Shear stress on snap ring


In [ ]:

t_snapring = 0.1 #inches

A_shear = math.pi*d_inner*t_snapring #in^2
F_injector_plate = math.pi*(d_inner/2)**2*p_chamber #lbf

tau_shear = F_injector_plate/A_shear

print(f"Applied shear stress: {tau_shear} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_shear}")
if tau_yield/tau_shear < n_safety:
    print("FAIL!")

Applied shear stress: 3000.0 psi
Yield shear strength: 17155.7679 psi
Safety factor: 5.7185893


# Von Mises Stress

Source: (https://en.wikipedia.org/wiki/Von_Mises_yield_criterion), State of stress: Principal stresses

In [41]:
sigma_van_mises = math.sqrt(0.5*((sigma_hoop-sigma_axial)**2+(sigma_axial-sigma_rad)**2+(sigma_rad-sigma_hoop)**2))

print(f"Applied van mises stress: {sigma_van_mises} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_van_mises}")
if tau_yield/tau_shear < n_safety:
    print("FAIL!")

Applied van mises stress: 2354.498925138321 psi
Yield strength: 29732.7 psi
Safety factor: 12.628037194051078


# Temperature Dependence

In [42]:
def yield_strength(T):
    """0.2% yield strength in psi at temperature T in deg C, linearly interpolated from datasheet values."""
    # [temp in C, yield strength in psi]
    temp_yield_data = np.array([[-253, 100e3],
                                [-196, 70e3],
                                [-79, 50e3],
                                [21, 35e3],
                                [205, 23e3],
                                [427, 19e3],
                                [650, 15.5e3],
                                [815, 13e3]])
    temps = temp_yield_data[:, 0]
    yields = temp_yield_data[:, 1]

    # np.interp clamps to the end values outside the data, which would overestimate strength above 815 C
    if np.any(np.asarray(T) < temps[0]) or np.any(np.asarray(T) > temps[-1]):
        raise ValueError(f"T = {T} C is outside the data range ({temps[0]:.0f} to {temps[-1]:.0f} C)")

    return np.interp(T, temps, yields)